# Laboratório 4: Classificadores com Scikit Learn e Pipelines

Bem-vindo ao Laboratório 4! Esta semana você utilizará alguns algoritmos implementados na biblioteca Scikit-Learn.

Os tópicos deste laboratório são abordados em [Logística](https://flaviovdf.io/icd-bradesco/14-Logistica/), [KNN](https://flaviovdf.io/icd-bradesco/15-KNN/) e [SKLearn](https://flaviovdf.io/icd-bradesco/16-Pratica/).

**Caso esteja usando collab, lembre-se, para fins didáticos, de desabilitar a assistência por AI nas configurações**


Primeiro, realize as importações executando a célula abaixo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

## Bank Loan Data
Esse exemplo visa a introdução de pipelines em sklearn.


Vamos utilizar o seguinte dataset que traz uma série de características sobre clientes e aponta caso o empréstimo realizado foi pago.
Carregando o dataset:

In [ ]:
bank_loan_df = pd.read_csv("https://raw.githubusercontent.com/flaviovdf/icd-bradesco/refs/heads/main/labs/lab04/loan_data.csv")

In [ ]:
bank_loan_df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


Queremos criar um modelo preditivo que, dadas as features de entrada, infere se o cliente irá quitar a dívida. Portanto, separamos o dataset em features de entrada X e variável a ser prevista Y. Além disso, amostramos o dataset para criar uma separação treino/teste.

In [ ]:
X = bank_loan_df.drop("loan_status",axis=1)
Y = bank_loan_df["loan_status"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)

Temos que algumas das colunas são categóricas (Exemplo: person_home_ownership ou loan_intent). Com isso, devemos buscar uma representação numérica para essas features serem passadas para o modelo. Uma forma bastante utilizada é o One-Hot encoding que transforma cada categoria em uma coluna binária. Além disso, para as colunas numéricas, é uma boa prática normalizar os dados a fim de se evitar que as diferenças escalas tornem o treinamento mais custoso. Felizmente, esses preprocessamentos já estão implementados em sklearn e podem ser vistos abaixo:

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numerical_cols)
])

preprocessor.fit_transform(X_train)

array([[ 1.        ,  0.        ,  0.        , ...,  0.57884523,
        -1.00034463,  0.58345167],
       [ 0.        ,  1.        ,  1.        , ..., -1.37324312,
        -1.00034463,  0.64295574],
       [ 0.        ,  1.        ,  1.        , ..., -0.22495585,
        -0.74224113, -1.1024968 ],
       ...,
       [ 0.        ,  1.        ,  0.        , ...,  0.92333141,
        -1.00034463,  1.41650857],
       [ 0.        ,  1.        ,  0.        , ..., -0.22495585,
        -1.00034463,  0.68262511],
       [ 1.        ,  0.        ,  0.        , ...,  0.23435905,
        -0.74224113,  1.23799637]])

Definido o preprocessamento, não queremos ter que preprocessar manualmente todos os dados a serem passados ao modelo. Essa é uma situação perfeita para se definir um pipeline de sklearn. Esse pipeline visa definir uma sequência de operações à qual todo dado passado para o modelo será submetido. No nosso caso, a primeira operação é o preprocessamento definido acima, e, após isso, a inferência pelo nosso modelo preditivo (especificamente uma regressão logística).

In [ ]:
classification_pipeline = make_pipeline(preprocessor, LogisticRegression(max_iter=100))

classification_pipeline.fit(X_train,Y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['person_gender',
                                                   'person_education',
                                                   'person_home_ownership',
                                                   'loan_intent',
                                                   'previous_loan_defaults_on_file']),
                                                 ('num', StandardScaler(),
                                                  ['person_age',
                                                   'person_income',
                                                   'person_emp_exp',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'cb_person_cred_hist_length',
                                                   'credit_score'])])),
                ('logisticregression', LogisticRegression())])

Estando o modelo treinado, podemos obter inferências sobre os dados (.predict)

In [ ]:
classification_pipeline.predict(X_test.iloc[[6]])

array([0])

Podemos também obter a acurácia do modelo no dataset de teste

In [ ]:
classification_pipeline.score(X_test,Y_test)

0.8962222222222223

## Identificação de malignidade em tumores de mama - UCI ML Breast Cancer Wisconsin (Diagnostic)

Neste trabalho você deverá realizar uma comparação entre o k-Nearest Neighbors (KNN) e a Regressão Logística para classificação de pacientes com tumores na mama (maligno = 1 vs benigno = 0). Para isso, usaremos o dataset de câncer de mama de UCI (já embutido no sklearn).

Você não precisa implementar os métodos, já que estão disponíveis na biblioteca scikit-learn da linguagem Python. Se necessário, pode fazer mais importações de bibliotecas.

Esse dataset está na biblioteca sklearn:

In [ ]:
# Importando o dataset
from sklearn.datasets import load_breast_cancer
# Baixando o dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data,columns =[cancer.feature_names])

In [ ]:
# Mostra uma descrição do dataset
print(cancer.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [ ]:
#Para acessar as labels
cancer.target

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

##Exercício 1:
Crie uma separação do dataset em conjuntos de treino de validação.

**Dica:** Use a função train_test_split para amostrar e separar treino e validação. Documentação da função [aqui](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)


In [ ]:
X = cancer.data #input
Y = cancer.target #label
# YOUR CODE HERE

##Exercício 2:
Use a regressão logística implementada no SciKit-Learn para classificar pacientes com e sem câncer.

Para a questão, faça as seguintes tarefas:


*   Com o parametro max_iter=100, realize o treinamento do modelo (.fit)
*   Obtenha a acurácia do modelo nos dados de validação (.score)


Perceba pelo Warning que o modelo não foi capaz de convergir (se ajustar de maneira estável) aos dados de treinamento dentro das 100 iterações.


In [ ]:
# YOUR CODE HERE

Uma forma de melhorar esse aspecto é normalizar os dados, assim todas as features (colunas) serão dotadas de média 0 e desvio padrão igual a 1. Isso visa tratar problemas gerados pela existência de diferentes escalas nos dados.

##Exercício 3:
Obtenha uma cópia normalizada dos dados.

**Dica:** Utilize StandardScaler de sklearn. Referência [aqui](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# YOUR CODE HERE

Podemos agora treinar um novo classificador com os dados normalizados. Entretanto há uma forma melhor de se implementar a execução de uma sequência de operações do scikit-learn: os pipelines.

##Exercício 4:
Crie e execute um pipeline para normalizar os dados e, em seguida, ajustar o modelo. Utilize novamente max_iter=100. Após ajustar o modelo obtenha sua acurácia no dataset de validação.

**Dica:** Utilize a função make_pipeline

In [ ]:
# YOUR CODE HERE

##Exercício 05:
Use uma KNN para classificar as pacientes com e sem câncer.

Para a questão, faça as seguintes tarefas:

*   Crie um pipeline que normaliza os dados e define o classificador.
*   Realize os experimentos avaliando apenas nos dados separados para validação
*   Escolha e reporte resultados com 3 números de vizinhos diferentes (k)
*   Retorne a acurácia dos modelos nos dados


In [ ]:
K = 3 # mude K aqui

In [ ]:
# YOUR CODE HERE

##Exercício 6 (Extra):
Obtenha outras métricas para os resultados dos classificadores tais como acurácia, precisão, revocação e f1. Referência [aqui](https://flaviovdf.io/icd-bradesco/16-Pratica/)

**Dica** Essas métricas estão implementadas no sklearn.

In [ ]:
# YOUR CODE HERE

##Exercício 7 (Extra):
Avalie os modelos utilizando validação cruzada. Referência [aqui](https://medium.com/@edubrazrabello/cross-validation-avaliando-seu-modelo-de-machine-learning-1fb70df15b78) e [aqui](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)

**Dica:** Utilize a função cross_validate. Essa função retorna arrays com metricas apuradas em cada rodada da validação cruzada.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

##Exercício 8 (Extra):
Avalie o desempenho dos classificadores em diferentes amostras do dataset.

Passos:
*   Criar um loop que se repete N vezes
*   Amostrar uma divisão treino/validação a cada iteração
*   Avaliar o modelo e armazenar as métricas em um vetor
*   Após repetir esse processo N vezes, plotar histogramas das métricas em ambos modelos


O que a distribuição das métricas te diz? Qual modelo é melhor?

In [ ]:
N = 500 # escolha N aqui

In [ ]:
# YOUR CODE HERE